In [1]:
# Standard library
import os
import random
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Torchvision
import torchvision
import torchvision.transforms as transforms

# Progress bars
from tqdm import tqdm

# Experiment tracking
import wandb


plt.style.use("default")
import sys
from dataclasses import dataclass

In [2]:
# 1. Clone your clean repository code from GitHub
REPO_URL = "https://github.com/K0NSTANT1N3/Facial-Expression-Recognition.git"
REPO_NAME = "Facial-Expression-Recognition"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repository already exists. Pulling latest updates...")
    os.system(f"cd {REPO_NAME} && git pull")

# 2. THE SECRET SAUCE: Add your cloned repo directory to Python's path
sys.path.append(os.path.abspath(REPO_NAME))

print("Successfully linked GitHub repository and adjusted system paths!")

Cloning repository...


Cloning into 'Facial-Expression-Recognition'...


Successfully linked GitHub repository and adjusted system paths!


In [3]:
import wandb

wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kende23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import wandb

run = wandb.init(
    project="fer2013",
    entity="kende23-n-a",
    name="test_run"
)

for epoch in range(5):
    run.log({
        "epoch": epoch,
        "accuracy": epoch * 0.1,
        "loss": 1/(epoch+1)
    })

run.finish()

accuracy,▁▃▅▆█
epoch,▁▃▅▆█
loss,█▄▂▁▁
accuracy,0.4
epoch,4
loss,0.2


In [50]:
@dataclass
class Config:
    csv_path: str = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 20

    num_classes: int = 7
    image_size: int = 48

    random_seed: int = 67


config = Config()

In [51]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.random_seed)

In [52]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [53]:
df = pd.read_csv(config.csv_path)

print(df.shape)
df.head()

(28709, 2)


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


In [54]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=config.random_seed,
    stratify=df["emotion"]
)

print(len(train_df))
print(len(val_df))

22967
5742


In [96]:
class_counts = train_df["emotion"].value_counts().sort_index()

print(class_counts)

emotion
0    3196
1     349
2    3277
3    5772
4    3864
5    2537
6    3972
Name: count, dtype: int64


In [98]:
class_weights = 1.0 / torch.tensor(
    class_counts.values,
    dtype=torch.float32
)

class_weights = class_weights / class_weights.sum()

print(class_weights)
class_weights = class_weights.to(device)

tensor([0.0686, 0.6282, 0.0669, 0.0380, 0.0567, 0.0864, 0.0552])


In [55]:
def pixels_to_image(pixel_string):
    pixels = np.fromstring(
        pixel_string,
        dtype=np.uint8,
        sep=" "
    )

    return pixels.reshape(48, 48)

In [56]:
class FERDataset(Dataset):

    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = pixels_to_image(row["pixels"])

        image = image.astype(np.float32) / 255.0

        image = torch.tensor(image).unsqueeze(0)

        label = torch.tensor(
            row["emotion"],
            dtype=torch.long
        )

        return image, label

In [57]:
train_dataset = FERDataset(train_df)
val_dataset = FERDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [58]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 1, 48, 48])
torch.Size([64])


In [85]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16), 
            nn.ReLU(),
            nn.Dropout2d(p=0.3),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), 
            nn.ReLU(),
            nn.Dropout2d(p=0.3),
            nn.MaxPool2d(kernel_size=2),

            # Block 3
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), 
            nn.ReLU(),
            nn.Dropout2d(p=0.3),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64 * 6 * 6, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x

In [86]:
model = SimpleCNN().to(device)

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout2d(p=0.3, inplace=False)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (6): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout2d(p=0.3, inplace=False)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): Dropout2d(p=0.3, inplace=False)
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (c

In [87]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)

Input shape : torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])


In [99]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [100]:
wandb.init(
    project="fer2013",
    entity="kende23-n-a",
    name="balance",
    config=vars(config)
)

wandb.watch(model)

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁▁▁
val_acc,▁▂▄▄▅▅▆▇▇▇▇▇████████
val_loss,█▆▅▄▃▃▂▂▂▂▁▁▁▂▁▂▁▁▂▂
epoch,20
train_acc,0.7017
train_loss,0.80236
val_acc,0.57524
val_loss,1.18398


In [101]:
def train_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() #?

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [102]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [103]:
print(next(model.parameters()).device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

cuda:0
True
Tesla T4


In [105]:
best_val_acc = 0

for epoch in range(config.epochs):

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch+1}/{config.epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_baseline_model.pt"
        )

Epoch 1/20 | Train Loss: 0.6880 | Train Acc: 0.7318 | Val Loss: 1.3337 | Val Acc: 0.5808
Epoch 2/20 | Train Loss: 0.6540 | Train Acc: 0.7445 | Val Loss: 1.3014 | Val Acc: 0.5742
Epoch 3/20 | Train Loss: 0.6522 | Train Acc: 0.7460 | Val Loss: 1.3523 | Val Acc: 0.5745
Epoch 4/20 | Train Loss: 0.6152 | Train Acc: 0.7611 | Val Loss: 1.3502 | Val Acc: 0.5747
Epoch 5/20 | Train Loss: 0.5900 | Train Acc: 0.7668 | Val Loss: 1.4023 | Val Acc: 0.5758
Epoch 6/20 | Train Loss: 0.5617 | Train Acc: 0.7807 | Val Loss: 1.3955 | Val Acc: 0.5702
Epoch 7/20 | Train Loss: 0.5571 | Train Acc: 0.7816 | Val Loss: 1.4706 | Val Acc: 0.5651
Epoch 8/20 | Train Loss: 0.5341 | Train Acc: 0.7864 | Val Loss: 1.4844 | Val Acc: 0.5728
Epoch 9/20 | Train Loss: 0.5319 | Train Acc: 0.7932 | Val Loss: 1.4720 | Val Acc: 0.5670
Epoch 10/20 | Train Loss: 0.5281 | Train Acc: 0.7946 | Val Loss: 1.4340 | Val Acc: 0.5630
Epoch 11/20 | Train Loss: 0.4918 | Train Acc: 0.8052 | Val Loss: 1.4657 | Val Acc: 0.5759
Epoch 12/20 | Train

In [106]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for x, y in val_loader:

        x = x.to(device)

        outputs = model(x)

        preds = outputs.argmax(dim=1)

        y_true.extend(y.numpy())
        y_pred.extend(preds.cpu().numpy())

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.48      0.43      0.45       799
           1       0.40      0.60      0.48        87
           2       0.44      0.43      0.43       820
           3       0.76      0.76      0.76      1443
           4       0.45      0.43      0.44       966
           5       0.70      0.72      0.71       634
           6       0.50      0.54      0.52       993

    accuracy                           0.57      5742
   macro avg       0.53      0.56      0.54      5742
weighted avg       0.57      0.57      0.57      5742



In [107]:
wandb.finish()

epoch,▁▁▂▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▂▂▂▃▃▄▄▅▅▅▅▆▆▆▇▇▇▇████
train_loss,█▇▇▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
val_acc,▄▁▅█▆▆▆▇▅▃▆▄▃▇▄▃▂▄▄▅▃▄▄
val_loss,▁▁▁▂▁▂▂▄▃▅▅▅▄▅▅▆▆▇▆█▇██
epoch,20
train_acc,0.84478
train_loss,0.40159
val_acc,0.56635
val_loss,1.60524
